# OCR — text extraction from meme images

Extracts overlaid text from meme images using EasyOCR.

**Why this matters:** the caption is half the multimodal signal. Unlike a social post, meme text is
*inside* the image and unavailable as metadata, so OCR quality directly bounds model performance.

**Why it is hard here:** stylized and heavily-outlined fonts, low contrast against busy backgrounds,
text spread across multiple panels with no reading order, and text rendered as part of the image
rather than as an overlay.

**Known limitation:** OCR quality was never measured against hand-transcribed ground truth — no
character error rate was computed, and no alternative engine was benchmarked. Correction was applied
downstream (`02_ocr_correction_t5.ipynb`) on the assumption it helped.


In [ ]:
import os
from google.colab import drive

In [ ]:
# Step 1: Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Define the root folder path inside Google Drive
root_folder = "/content/drive/MyDrive/LGBTQ Memes/Combined_1500"

In [ ]:
# Count the number of files in the root folder
file_count = 0
for _, _, files in os.walk(root_folder):
    file_count += len(files)

print(f"The number of files in '{root_folder}' is: {file_count}")


In [ ]:
# Count the number of files in the root folder
count = 0
for filename in os.listdir(root_folder):
    if os.path.isfile(os.path.join(root_folder, filename)):
        base, ext = os.path.splitext(filename)
        if ext.lower() in ['.png', '.jpg', '.jpeg', '.webp']:
            count += 1
            new_filename = f"img_{count}{ext.lower()}"
            os.rename(os.path.join(root_folder, filename), os.path.join(root_folder, new_filename))

print(f"Renamed {count} files in '{root_folder}'")


In [ ]:
count = 0
img_base_names = []
for filename in os.listdir(root_folder):
    if os.path.isfile(os.path.join(root_folder, filename)):
        base, ext = os.path.splitext(filename)
        print(base)
        count += 1
        img_base_names.append(base)

print(count, "names added to list")




In [ ]:
print(img_base_names)

In [ ]:
count = 0
img_base_names = []
for filename in os.listdir(root_folder):
    if os.path.isfile(os.path.join(root_folder, filename)):
        base, ext = os.path.splitext(filename)
        count += 1
        base_iter = f"img_{count}"
        if base_iter not in img_base_names:
            print(base_iter)

In [ ]:
import re

In [ ]:
# Extract numbers from the strings
extracted_numbers = set()
for name in img_base_names:
    match = re.search(r'\d+', name)  # Find any number in the string
    if match:
        num = int(match.group())
        if 1 <= num <= 1494:
            extracted_numbers.add(num)

# Find missing numbers
missing_numbers = set(range(1, 1495)) - extracted_numbers

# Print missing numbers
print("Missing numbers:", sorted(missing_numbers))

In [ ]:
# Step 4: Maintain a count variable and rename files
count = 1

for folder in subfolders:
    files = sorted(os.listdir(folder))  # Sort files alphabetically
    for file in files:
        file_path = os.path.join(folder, file)

        if os.path.isfile(file_path):  # Ensure it's a file
            ext = os.path.splitext(file)[1]  # Extract file extension
            new_filename = f"{count}{ext}"  # Create new filename

            new_path = os.path.join(folder, new_filename)
            os.rename(file_path, new_path)  # Rename file

            print(f"Renamed: {file} -> {new_filename}")
            count += 1


In [ ]:
!pip install easyocr

In [ ]:
import os
import cv2
import csv
import easyocr
import matplotlib.pyplot as plt
from google.colab import drive

In [ ]:
# Initialize OCR reader
reader = easyocr.Reader(['en'])  # Specify language

# Initialize CSV file
csv_filename = "/content/drive/MyDrive/ocr_results.csv"
csv_headers = ["img_idx", "sub_folder_path", "ocr_text"]

data_rows = []

In [ ]:
# Process each image in subfolders
for folder in subfolders:
    files = sorted(os.listdir(folder))  # Sort files alphabetically

    for file in files:
        file_path = os.path.join(folder, file)

        if os.path.isfile(file_path):  # Ensure it's a file
            img_idx = os.path.splitext(file)[0]  # Extract image index (filename without extension)

            # Step 4: Load and Display the Image
            img = cv2.imread(file_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for proper display

            plt.figure(figsize=(10, 6))
            plt.imshow(img)
            plt.axis('off')
            plt.show()

            # Step 5: Apply OCR using EasyOCR
            result = reader.readtext(file_path)

            # Merge OCR results into a single string separated by " "
            extracted_text = " ".join([detection[1] for detection in result])

            # Step 6: Store data in list
            relative_path = os.path.relpath(file_path, root_folder)  # Get relative path from root
            data_rows.append([img_idx, relative_path, extracted_text])

            print(f"Processed: {file} | OCR: {extracted_text[:50]}...")  # Show preview




In [ ]:
# Step 7: Write data to CSV
with open(csv_filename, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(csv_headers)  # Write headers
    writer.writerows(data_rows)  # Write rows

print(f"CSV saved at: {csv_filename}")